In [ ]:
import torch

from market.events import EventQueue, FillEvent
from pipeline.data_handler import DataHandler
from strategy.model import Model
from strategy.predict import Strategy
from strategy.risk.risk_manager import RiskManager
from strategy.position.portfolio import Portfolio
from market.exchange import Exchange
from backtest import performance_calculator as pf_calc
from utility import get_config, get_path, read_file

from strategy.environment import MarketEnv

In [ ]:
config = get_config.load()
start_date = config['pipeline']['start_date']
end_date = config['pipeline']['end_date']
portfolio_dir = config['path']['strategy']['portfolio']

In [ ]:
data = read_file.read_data('train', 'data')
type(data)
data.head()

In [ ]:
SYMBOLS = [s.split('/')[0] for s in config['pipeline']['symbols']]
model_dir = get_path.absolute(config['path']['strategy']['model'])
portfolio_dir = get_path.absolute(config['path']['strategy']['portfolio'])
get_path.check(model_dir)
get_path.check(portfolio_dir)
seq_len = config['strategy']['sequence_length']
min_amt = config['backtest']['minimum_amount']
txn_cost = config['backtest']['transaction_cost_fraction']
capital = config['backtest']['capital']
bankruptcy_fraction = config['strategy']['bankruptcy_fraction']
slippage = config['strategy']['slippage_cost_fraction']
slm = config['strategy']['stop_loss_multiple']
slp = config['strategy']['stop_loss_fraction']
tpm = config['strategy']['take_profit_multiple']
tpp = config['strategy']['take_profit_fraction']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

market_updates = 0
event_queue = EventQueue()
data_handler = DataHandler(data, SYMBOLS, start_date, end_date, portfolio_dir)
# model = Model(SYMBOLS, model_dir)
# strategy = Strategy(data_handler, model, SYMBOLS, seq_len, device)
risk_m = RiskManager(data_handler, SYMBOLS, min_amt, slm, slp, tpm, tpp)
portfolio = Portfolio(SYMBOLS, data_handler, risk_m, capital, slippage, txn_cost, min_amt, bankruptcy_fraction, portfolio_dir)
exchange = Exchange(SYMBOLS, txn_cost, min_amt)
performance_calculator = pf_calc.PerformanceCalculator(capital)

In [ ]:
# Environment Variables

max_episode_steps = 100
timestamp = 0
description = "test"
cash_delta = 10

In [ ]:
env = MarketEnv(data_handler, portfolio, max_episode_steps = max_episode_steps)

In [ ]:
action = FillEvent(timestamp, description, cash_delta, txn_cost)

In [ ]:
for i in range(5):
    data_handler.update_candles()

In [ ]:
from gymnasium.utils.env_checker import check_env

episode_complete = False
while not episode_complete:
    observation, reward, terminated, truncated, info = env.step(action)
    print(reward)
    if truncated or terminated:
        episode_complete = True
        env.reset()


In [ ]:
from gymnasium.utils.env_checker import check_env

# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")